## Merge confidence values with spatial_obl values to sort them into categories

Datapoints get :

- level: n80, n90, n70, n10, n20, n30, -

- olulisus: p-value

- annotated word count (form, ekilex_tag=given tag)

- not annotated word count (form, ekilex_tag is null)

- unique lemma count (ekilex_tag=given tag)


Final LINE_DATA_TABLE2 can be used to later extract info for gpt

In [1]:
import sqlite3
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from scipy.stats import binom
import copy

In [2]:
DATABASE = "../../drive_data/v33_koondkorpus_sentences_verb_pattern_obl_20241002-130310.db"

TAG = "ELT"

# obl table etc
OBL_TABLE = "spatial_obl"
TAG_COLUMN = "tags"
VERB_COUNTS_TABLE = f"verb_case_counts_{TAG}"
VERB_CASE_TABLE = f"verb_case_log_{TAG}"

# confidence values
CONFIDENCE_TABLE = "confidence_values"

# tables to save verb+case info with n class
LINE_DATA_TABLE = f"spatial_obl_{TAG}_n_class_wide"
LINE_DATA_TABLE2 = f"spatial_obl_{TAG}_n_class"

## Data tables

In [37]:
conn = sqlite3.connect(DATABASE)
cursor = conn.cursor()

In [4]:
# confidence values
query = f"SELECT * FROM {CONFIDENCE_TABLE}"
lines_df = pd.read_sql(query, conn)

In [5]:
lines_df

,x,y_pos80,log2_x,y_neg80,log2_y_pos80,y_pos90,y_neg90,log2_y_pos90,y_pos70,y_neg70,...,log2_y_pos90_p1,log2_y_pos90_m1,log2_y_pos70_p1,log2_y_pos70_m1,log2_y_pos20_p1,log2_y_pos20_m1,log2_y_pos10_p1,log2_y_pos10_m1,log2_y_pos30_p1,log2_y_pos30_m1
0,1,1,0.000000,0,inf,1,0,inf,0,1,...,NaN,-inf,inf,NaN,inf,NaN,inf,NaN,NaN,-inf
1,2,2,1.000000,0,inf,2,0,inf,0,2,...,NaN,0.000000,0.000000,NaN,0.000000,NaN,0.000000,NaN,NaN,0.000000
2,3,3,1.584963,0,inf,3,0,inf,1,2,...,NaN,1.000000,1.000000,-inf,-1.000000,NaN,-1.000000,NaN,NaN,1.000000
3,4,4,2.000000,0,inf,4,0,inf,1,3,...,NaN,1.584963,0.000000,-inf,-1.584963,NaN,-1.584963,NaN,NaN,1.584963
4,5,5,2.321928,0,inf,5,0,inf,2,3,...,NaN,2.000000,0.584963,-2.000000,-2.000000,NaN,-2.000000,NaN,inf,0.584963
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4994,4995,4043,12.286269,952,2.086393,4531,464,3.287633,3443,1552,...,3.291064,3.284208,1.150886,1.148189,-2.082651,-2.086393,-3.280791,-3.287633,-1.146841,-1.149538
4995,4996,4044,12.286558,952,2.086750,4532,464,3.287951,3444,1552,...,3.291382,3.284527,1.151305,1.148608,-2.083008,-2.086750,-3.281109,-3.287951,-1.147261,-1.149957
4996,4997,4045,12.286847,952,2.087106,4533,464,3.288269,3445,1552,...,3.291700,3.284845,1.151724,1.149027,-2.083365,-2.087106,-3.281428,-3.288269,-1.147680,-1.150375
4997,4998,4046,12.287135,952,2.087463,4534,464,3.288588,3445,1553,...,3.292018,3.285164,1.150794,1.148099,-2.083722,-2.087463,-3.281746,-3.288588,-1.146752,-1.149446


In [6]:
# table with verb+case data and unique lemma number
query = f"SELECT * FROM {VERB_CASE_TABLE}"
verb_case_log = pd.read_sql(query, conn)

In [7]:
verb_case_log

,verb,verb_compound,morph_case,log2_tag,log2_annotation,verb_case_count,unique_lemmas,log2_unique_lemmas,synset_count
0,aasima,,ad,9.965784,-2.807355,8,1,0.000000,2-3
1,aasima,,in,0.000000,-1.321928,7,2,1.000000,2-3
2,abielluma,,abl,9.965784,-5.658211,103,2,1.000000,1
3,abielluma,,ad,6.686501,-0.880761,1182,78,6.285402,1
4,abielluma,,adit,-3.584963,0.378512,23,4,2.000000,1
...,...,...,...,...,...,...,...,...,...
20935,šokeerima,,ad,9.965784,-0.980371,110,25,4.643856,2-3
20936,šokeerima,,all,9.965784,-2.321928,6,1,0.000000,2-3
20937,šokeerima,,el,-9.965784,-1.584963,16,4,2.000000,2-3
20938,šokeerima,,in,9.965784,-0.536053,49,16,4.000000,2-3


## merge

In [8]:
new_df = pd.merge(verb_case_log, lines_df, left_on='unique_lemmas', right_on='x')
new_df

,verb,verb_compound,morph_case,log2_tag,log2_annotation,verb_case_count,unique_lemmas,log2_unique_lemmas,synset_count,x,...,log2_y_pos90_p1,log2_y_pos90_m1,log2_y_pos70_p1,log2_y_pos70_m1,log2_y_pos20_p1,log2_y_pos20_m1,log2_y_pos10_p1,log2_y_pos10_m1,log2_y_pos30_p1,log2_y_pos30_m1
0,aasima,,ad,9.965784,-2.807355,8,1,0.000000,2-3,1,...,NaN,-inf,inf,NaN,inf,NaN,inf,NaN,NaN,-inf
1,abistama,,all,-9.965784,-3.906891,16,1,0.000000,2-3,1,...,NaN,-inf,inf,NaN,inf,NaN,inf,NaN,NaN,-inf
2,aeglustama,,all,-9.965784,-3.000000,9,1,0.000000,1,1,...,NaN,-inf,inf,NaN,inf,NaN,inf,NaN,NaN,-inf
3,aerutama,,in,9.965784,-1.736966,13,1,0.000000,1,1,...,NaN,-inf,inf,NaN,inf,NaN,inf,NaN,NaN,-inf
4,aevastama,,in,9.965784,-2.584963,7,1,0.000000,1,1,...,NaN,-inf,inf,NaN,inf,NaN,inf,NaN,NaN,-inf
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20935,õnnestuma,,ad,-0.779489,-1.777094,22274,1313,10.358651,2-3,1313,...,3.422666,3.394726,1.088271,1.078182,-2.160544,-2.175303,-3.380922,-3.408640,-1.073148,-1.083223
20936,õppima,,in,6.751420,-0.004661,11763,530,9.049849,2-3,530,...,3.614710,3.538420,1.020464,0.995919,-2.251225,-2.289507,-3.501513,-3.576139,-0.983698,-1.008174
20937,ütlema,,ad,6.664753,0.326842,22466,415,8.696968,2-3,415,...,3.681824,3.581201,0.989583,0.958481,-2.276518,-2.326104,-3.533035,-3.630766,-0.943010,-0.974005
20938,ütlema,,all,0.384871,-1.252183,57846,1131,10.143383,2-3,1131,...,3.446953,3.414108,1.075288,1.063616,-2.175388,-2.192645,-3.397915,-3.430453,-1.057793,-1.069448


## Assign n class for each obl verb+case

if >= 90

<90 & >=80

<= 70 & >= 30

<20

<10


In [9]:
def get_level(row):
    if row['log2_tag'] >= row[f"log2_y_pos90"]: #kõrgemal 90 joonest
        return "n90"
    elif row['log2_tag'] >= row[f"log2_y_pos80"] and row['log2_tag'] < row[f"log2_y_pos90"]: #kõrgemal 80 joonest
        return "n80"
    elif row['log2_tag'] >= 0 and row['log2_tag'] <= row[f"log2_y_pos70"]: #madalamal 70 ja kõrgemal 0 joonest
        return "n70"
    elif row['log2_tag'] >= row[f"log2_y_pos30"] and row['log2_tag'] <= 0: #madalamal 0 ja kõrgemal 30 joonest
        return "n30"
    elif row['log2_tag'] <= row[f"log2_y_pos20"] and row['log2_tag'] > row[f"log2_y_pos10"]: #madalamal 20 joonest
        return "n20"
    elif row['log2_tag'] <= row[f"log2_y_pos10"]: #madalamal 10 joonest
        return "n10"
    else:
        return "-"

In [10]:
new_df["level"] = new_df.apply(get_level, axis=1)

## not_ann_words ja ann_words 

(Not lemmas but "form" from spatial_obl table where ekilex_tag is null or not)

In [11]:
query = f"""
        SELECT verb, verb_compound, morph_case, count(form) as not_ann_words
        FROM {OBL_TABLE}
        WHERE {TAG_COLUMN} = ''
        GROUP BY verb, verb_compound, morph_case;
        """

df_notag = pd.read_sql(query, conn)
df_notag

,verb,verb_compound,morph_case,not_ann_words
0,0muutuma,,el,1
1,0olema,,ad,1
2,0olema,,el,2
3,0olema,,in,2
4,10halama,,in,1
...,...,...,...,...
66037,šveitsima,,in,6
66038,žestikuleerima,,ad,1
66039,žongleerima,,ad,3
66040,žongleerima,,in,2


In [12]:
# Merge with df_log based on verb, verb_compound, morph_case
new_df = new_df.merge(df_notag, on=['verb', 'verb_compound', 'morph_case'], how='left')

In [38]:
#tags = f"('{TAG}')"

query = f"""
        SELECT verb, verb_compound, morph_case, count(form) as ann_words
        FROM {OBL_TABLE}
        WHERE {TAG_COLUMN} like '%|{TAG}|%'
        GROUP BY verb, verb_compound, morph_case;
        """

df_tag = pd.read_sql(query, conn)
df_tag

,verb,verb_compound,morph_case,ann_words
0,1811vastama,,ad,1
1,1811vastama,,el,1
2,1991hääletama,,ad,1
3,1991pooldama,,ad,1
4,21olema,,in,1
...,...,...,...,...
31902,šokeerima,,in,20
31903,šoppama,,in,2
31904,šveitsima,,ad,1
31905,švipsima,,ad,1


In [14]:
# Merge with df_log based on verb, verb_compound, morph_case
new_df = new_df.merge(df_tag, on=['verb', 'verb_compound', 'morph_case'], how='left')

## How many unique lemmas are annotated

In [16]:
#tags = f"('{TAG}')"

query = f"""
    SELECT 
        verb, 
        verb_compound,
        morph_case, 
        COUNT(DISTINCT lemma) AS ann_unique_lemmas
    FROM {OBL_TABLE}
    WHERE {TAG_COLUMN} like '%|{TAG}|%'
    GROUP BY verb, verb_compound, morph_case
"""

df_ul_tag = pd.read_sql(query, conn)
df_ul_tag

,verb,verb_compound,morph_case,ann_unique_lemmas
0,1811vastama,,ad,1
1,1811vastama,,el,1
2,1991hääletama,,ad,1
3,1991pooldama,,ad,1
4,21olema,,in,1
...,...,...,...,...
31902,šokeerima,,in,16
31903,šoppama,,in,2
31904,šveitsima,,ad,1
31905,švipsima,,ad,1


In [17]:
# Merge with df_log based on verb, verb_compound, morph_case
new_df = new_df.merge(df_ul_tag, on=['verb', 'verb_compound', 'morph_case'], how='left')

## How many unique lemmas are not annotated

In [18]:
query = f"""
    SELECT 
        verb, 
        verb_compound,
        morph_case, 
        COUNT(DISTINCT lemma) AS not_ann_unique_lemmas
    FROM {OBL_TABLE}
    WHERE {TAG_COLUMN} = ''
    GROUP BY verb, verb_compound, morph_case
"""

df_ul_ntag = pd.read_sql(query, conn)
df_ul_ntag

,verb,verb_compound,morph_case,not_ann_unique_lemmas
0,0muutuma,,el,1
1,0olema,,ad,1
2,0olema,,el,2
3,0olema,,in,2
4,10halama,,in,1
...,...,...,...,...
66037,šveitsima,,in,6
66038,žestikuleerima,,ad,1
66039,žongleerima,,ad,2
66040,žongleerima,,in,2


In [19]:
new_df = new_df.merge(df_ul_ntag, on=['verb', 'verb_compound', 'morph_case'], how='left')

## olulisus (p-value)


"""
def get_min_success(n=100, p=0.8, kv=0.05):

    #n = 100       # number of trials
    #p = 0.8       # null hypothesis success rate

    # Find smallest k such that P(X ≥ k) < 0.05
    for k in range(n + 1):
        if binom.sf(k - 1, n, p) <= kv: #kv=0.05 95% puhul, 70% puhul peaks olema 0.05 asemel 0.95
            #print(f"Minimum k: {k}")
            #break
            return k
    
    #return np.nan
    return n
"""

def kv_for_datapoint(n, log2_ratio, p=0.8): #(log2_unique_lemmas->unique_lemmas, log2_tag, p)
    #n=2**log2_x
    # Compute observed successes from log2(y_pos/y_neg)
    ratio = 2 ** log2_ratio
    k_obs = n * ratio / (1 + ratio)
    # Compute probability P(X >= k_obs)
    kv_obs = binom.sf(int(round(k_obs)) - 1, int(round(n)), p)
    return n, k_obs, kv_obs

### Example: line point at (11.16, 2.13) corresponds to 5% line
### Above-point at (11.16, 3.06)
n_line, k_line, kv_line = kv_for_datapoint(1534, 2.163039, p=0.8)
n_obs, k_obs, kv_obs = kv_for_datapoint(1534, 2.236495, p=0.8) # (log2_unique_lemmas, log2_tag, p)

print(f"n (unique lemmas) = {n_line:.0f}")
print(f"Critical line point: k = {k_line:.0f} (y_pos80), kv = {kv_line:.4f}")
print(f"Above point: k = {k_obs:.0f}, kv = {kv_obs:.6f}")

**CDF = left side = P(X ≤ k)**

**SF = right side = P(X > k)**

**SF(k-1) = P(X ≥ k)**

**use CDF for lower tail, SF for upper tail**

In [20]:
mapping_level = {"n80": 0.8, "n90": 0.9, "n70":0.7, "n20": 0.2, "n10":0.1, "n30": 0.3}

def kv_for_datapoint(row): #(log2_unique_lemmas->unique_lemmas, log2_tag, p)
    #n=2**log2_x : log2_x = log2_unique_lemmas
    # n : unique_lemmas
    # log2_ratio : log2_tag
    # p = 0.8 kui n80
    
    if row["level"] != "-":
        n = row["unique_lemmas"]
        log2_ratio = row["log2_tag"]
        p = mapping_level[row["level"]]

        # Compute observed successes from log2(y_pos/y_neg)
        ratio = 2 ** log2_ratio
        k_obs = n * ratio / (1 + ratio)
        # Compute probability P(X >= k_obs), üleval pool joont
        if p>0.5:
            kv_obs = binom.sf(int(round(k_obs)) - 1, int(round(n)), p)
        else: # all pool joont
            kv_obs = binom.cdf(int(round(k_obs)), int(round(n)), p)
        #return n, k_obs, kv_obs
        return round(kv_obs, 5)
    else:
        return "-"

In [21]:
new_df["olulisus"] = new_df.apply(kv_for_datapoint, axis=1)

In [22]:
new_df = new_df.rename(columns={'log2_tag': 'log2_ratio'})

In [23]:
new_df

,verb,verb_compound,morph_case,log2_ratio,log2_annotation,verb_case_count,unique_lemmas,log2_unique_lemmas,synset_count,x,...,log2_y_pos10_p1,log2_y_pos10_m1,log2_y_pos30_p1,log2_y_pos30_m1,level,not_ann_words,ann_words,ann_unique_lemmas,not_ann_unique_lemmas,olulisus
0,aasima,,ad,9.965784,-2.807355,8,1,0.000000,2-3,1,...,inf,NaN,NaN,-inf,-,7.0,1.0,1.0,5.0,-
1,abistama,,all,-9.965784,-3.906891,16,1,0.000000,2-3,1,...,inf,NaN,NaN,-inf,-,15.0,NaN,NaN,12.0,-
2,aeglustama,,all,-9.965784,-3.000000,9,1,0.000000,1,1,...,inf,NaN,NaN,-inf,-,8.0,NaN,NaN,7.0,-
3,aerutama,,in,9.965784,-1.736966,13,1,0.000000,1,1,...,inf,NaN,NaN,-inf,-,10.0,3.0,1.0,5.0,-
4,aevastama,,in,9.965784,-2.584963,7,1,0.000000,1,1,...,inf,NaN,NaN,-inf,-,6.0,1.0,1.0,6.0,-
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20935,õnnestuma,,ad,-0.779489,-1.777094,22274,1313,10.358651,2-3,1313,...,-3.380922,-3.408640,-1.073148,-1.083223,n30,17243.0,1852.0,487.0,2522.0,1.0
20936,õppima,,in,6.751420,-0.004661,11763,530,9.049849,2-3,530,...,-3.501513,-3.576139,-0.983698,-1.008174,n90,5891.0,5818.0,502.0,985.0,0.0
20937,ütlema,,ad,6.664753,0.326842,22466,415,8.696968,2-3,415,...,-3.533035,-3.630766,-0.943010,-0.974005,n90,9966.0,12378.0,375.0,1162.0,0.0
20938,ütlema,,all,0.384871,-1.252183,57846,1131,10.143383,2-3,1131,...,-3.397915,-3.430453,-1.057793,-1.069448,n70,40742.0,9686.0,231.0,2337.0,1.0


In [24]:
new_df.to_sql(LINE_DATA_TABLE, conn, if_exists="replace", index=False)

20940

## What was the given tag number and other tag number 

In [25]:
# interesting columns are up to "not_annotated", the rest is for extra information

query = f"""SELECT tbl1.verb, tbl1.verb_compound, tbl1.morph_case, 
        log2_ratio, unique_lemmas, level, ann_unique_lemmas,
        not_ann_unique_lemmas, olulisus, my_tag, other_tags, annotated, not_annotated,
        log2_annotation, tbl1.verb_case_count, 
        log2_unique_lemmas,synset_count, x, y_pos80, log2_x, y_neg80, log2_y_pos80,
       y_pos90, y_neg90, log2_y_pos90, y_pos70, y_neg70,log2_y_pos70, y_pos30, y_neg30, log2_y_pos30, y_pos20,
       y_neg20, log2_y_pos20, y_pos10, y_neg10, log2_y_pos10, not_ann_words, ann_words
       --,{VERB_COUNTS_TABLE}.verb_case_count as loc_verb_case_count

            FROM {LINE_DATA_TABLE} as tbl1
            left join 
            {VERB_COUNTS_TABLE} 
            on 
            tbl1.verb = {VERB_COUNTS_TABLE}.verb and
            tbl1.verb_compound = {VERB_COUNTS_TABLE}.verb_compound and
            tbl1.morph_case = {VERB_COUNTS_TABLE}.morph_case
            """

df = pd.read_sql(query, conn)
df

,verb,verb_compound,morph_case,log2_ratio,unique_lemmas,level,ann_unique_lemmas,not_ann_unique_lemmas,olulisus,my_tag,...,y_neg30,log2_y_pos30,y_pos20,y_neg20,log2_y_pos20,y_pos10,y_neg10,log2_y_pos10,not_ann_words,ann_words
0,aasima,,ad,9.965784,1,-,1.0,5.0,-,1,...,0,inf,0,1,-inf,0,1,-inf,7.0,1.0
1,abistama,,all,-9.965784,1,-,NaN,12.0,-,0,...,0,inf,0,1,-inf,0,1,-inf,15.0,NaN
2,aeglustama,,all,-9.965784,1,-,NaN,7.0,-,0,...,0,inf,0,1,-inf,0,1,-inf,8.0,NaN
3,aerutama,,in,9.965784,1,-,1.0,5.0,-,3,...,0,inf,0,1,-inf,0,1,-inf,10.0,3.0
4,aevastama,,in,9.965784,1,-,1.0,6.0,-,1,...,0,inf,0,1,-inf,0,1,-inf,6.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20935,õnnestuma,,ad,-0.779489,1313,n30,487.0,2522.0,1.0,1852,...,891,-1.078182,239,1074,-2.167911,114,1199,-3.394726,17243.0,1852.0
20936,õppima,,in,6.751420,530,n90,502.0,985.0,0.0,5818,...,353,-0.995919,91,439,-2.270282,42,488,-3.538420,5891.0,5818.0
20937,ütlema,,ad,6.664753,415,n90,375.0,1162.0,0.0,12378,...,274,-0.958481,70,345,-2.301170,32,383,-3.581201,9966.0,12378.0
20938,ütlema,,all,0.384871,1131,n70,231.0,2337.0,1.0,9686,...,765,-1.063616,204,927,-2.184000,97,1034,-3.414108,40742.0,9686.0


In [ ]:
# verb_case_count peaks olema sama, mis loc_verb_case_count, ehk võib võtta ühe
# ann_words peaks olema sama, mis my_tag (NaN vs 0 ka)
# not_ann_words peaks olema sama, mis not_annotated (Nan vs 0 ka)


In [26]:
df[df["verb"]=="käima"]

,verb,verb_compound,morph_case,log2_ratio,unique_lemmas,level,ann_unique_lemmas,not_ann_unique_lemmas,olulisus,my_tag,...,y_neg30,log2_y_pos30,y_pos20,y_neg20,log2_y_pos20,y_pos10,y_neg10,log2_y_pos10,not_ann_words,ann_words
1037,käima,kallal,ad,-9.965784,1,-,NaN,7.0,-,0,...,0,inf,0,1,-inf,0,1,-inf,7.0,NaN
1038,käima,kannul,ad,9.965784,1,-,1.0,4.0,-,1,...,0,inf,0,1,-inf,0,1,-inf,7.0,1.0
1039,käima,kinni,in,9.965784,1,-,1.0,5.0,-,1,...,0,inf,0,1,-inf,0,1,-inf,5.0,1.0
1040,käima,ringi,all,-9.965784,1,-,NaN,4.0,-,0,...,0,inf,0,1,-inf,0,1,-inf,5.0,NaN
1041,käima,sisse,el,9.965784,1,-,1.0,14.0,-,1,...,0,inf,0,1,-inf,0,1,-inf,21.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20593,käima,,adit,2.342888,124,-,96.0,421.0,-,208,...,77,-0.712198,18,106,-2.557995,7,117,-4.063010,1531.0,208.0
20598,käima,,all,-1.170483,461,-,156.0,856.0,-,287,...,305,-0.967263,78,383,-2.295798,36,425,-3.561394,3889.0,287.0
20599,käima,,el,0.949030,837,n70,507.0,1979.0,0.99588,1446,...,563,-1.038959,149,688,-2.207096,70,767,-3.453800,5808.0,1446.0
20600,käima,,in,5.571852,2220,n90,2051.0,4636.0,0.0,21690,...,1517,-1.109624,413,1807,-2.129383,199,2021,-3.344229,27381.0,21690.0


In [24]:
# log2_ratio = np.log2((my_tag/annotated)/(other_tags/annotated))
# np.log2((167/295)/(128/295))

0.38370429247405224

## salvestada andmebaasi

In [27]:
#new_df.to_sql("lines_class_info3", conn, if_exists="replace", index=False)
df.to_sql(LINE_DATA_TABLE2, conn, if_exists="replace", index=False)

20940

## uuesti sisse lugemine

In [28]:
# log2_unique_lemmas on sama, mis log2_x
# log2_tag on andmepunkti y 

query = f"""SELECT verb, verb_compound, morph_case, log2_ratio, unique_lemmas, level, 
            ann_unique_lemmas, not_ann_unique_lemmas, olulisus,
            my_tag, other_tags, annotated, not_annotated
            FROM {LINE_DATA_TABLE2}
            """

df = pd.read_sql(query, conn)

In [29]:
df

,verb,verb_compound,morph_case,log2_ratio,unique_lemmas,level,ann_unique_lemmas,not_ann_unique_lemmas,olulisus,my_tag,other_tags,annotated,not_annotated
0,aasima,,ad,9.965784,1,-,1.0,5.0,-,1,0,1,7
1,abistama,,all,-9.965784,1,-,NaN,12.0,-,0,1,1,15
2,aeglustama,,all,-9.965784,1,-,NaN,7.0,-,0,1,1,8
3,aerutama,,in,9.965784,1,-,1.0,5.0,-,3,0,3,10
4,aevastama,,in,9.965784,1,-,1.0,6.0,-,1,0,1,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...
20935,õnnestuma,,ad,-0.779489,1313,n30,487.0,2522.0,1.0,1852,3179,5031,17243
20936,õppima,,in,6.751420,530,n90,502.0,985.0,0.0,5818,54,5872,5891
20937,ütlema,,ad,6.664753,415,n90,375.0,1162.0,0.0,12378,122,12500,9966
20938,ütlema,,all,0.384871,1131,n70,231.0,2337.0,1.0,9686,7418,17104,40742


In [30]:
df2 = df[df["level"]!= "-"]

In [31]:
df3 = df2[df2["level"]=="n80"]

In [32]:
df3 = df3.sort_values(["olulisus"])

In [33]:
df3

,verb,verb_compound,morph_case,log2_ratio,unique_lemmas,level,ann_unique_lemmas,not_ann_unique_lemmas,olulisus,my_tag,other_tags,annotated,not_annotated
20933,võtma,,ad,3.328994,700,n80,475.0,1565.0,0.0,3889,387,4276,12272
20538,kõnelema,,in,4.220048,120,n80,112.0,321.0,0.0,205,11,216,853
20543,korduma,,in,4.292782,113,n80,106.0,281.0,0.0,196,10,206,459
15697,vastutama,,in,4.123382,135,n80,126.0,227.0,0.0,244,14,258,642
17791,leidma,,el,2.533148,1120,n80,861.0,3239.0,0.0,3473,600,4073,11238
...,...,...,...,...,...,...,...,...,...,...,...,...,...
20566,kutsuma,,adit,3.519553,138,n80,117.0,217.0,9.0e-05,539,47,586,2003
18301,hukkuma,kokku,in,9.965784,42,n80,42.0,44.0,9.0e-05,106,0,106,86
20453,täitma,,ad,3.459432,152,n80,144.0,393.0,9.0e-05,451,41,492,1751
20233,jooksma,,in,2.899327,313,n80,272.0,660.0,9.0e-05,761,102,863,1825


In [39]:
conn.close()